## Actividad 3_20: Perros y gatos
<div style="border-style:groove;border-width:thin;padding:10px">
En esta actividad vamos a utilizar las técnicas de redes neuronales y deep learning que hemos visto en clase para enseñar a este software a diferenciar entre perros y gatos.

Para ello vamos a cargar los datos y etiquetarlos, a lanzar un Random Forest Classifier para establecer un punto de partida que debemos mejorar y después, vamos a tratar de solucionar el problema con una red neuronal convencional.
</div>

In [2]:
# Datos y preprocesamiento
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Modelos de Clasificación
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Modelos de Regresión
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from tensorflow import keras

# Métricas de Clasificación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Métricas de Regresión
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

I0000 00:00:1776699687.851300   13689 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776699691.888281   13689 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776699700.230462   13689 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./PetImages')

photos =  []
labels = []

In [4]:
for idx,folder in enumerate(folders):
    for file in listdir('./PetImages/'+folder):
        photo = load_img('./PetImages/'+folder+'/' + file, target_size=(64, 64)) 
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

/home/ciabd14/anaconda3/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


0
1


In [5]:
# max_por_carpeta = 2000

# for idx, folder in enumerate(folders):
#     contador = 0

#     for file in listdir('./PetImages/' + folder):
#         if contador >= max_por_carpeta:
#             break

#         photo = load_img('./PetImages/' + folder + '/' + file, target_size=(128, 128))
#         photo = img_to_array(photo)

#         photos.append(photo)
#         labels.append(float(idx))

#         contador += 1
#         del photo

#     print(f"Carpeta {idx}: {contador} imágenes")

In [6]:
photos = np.array(photos)
labels = np.array(labels)

In [7]:
photos.shape

(24998, 64, 64, 3)

In [8]:
X = photos
y = labels

In [9]:
X = X/255.0

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

## RANDOM FOREST

In [11]:
X_train_reshape = X_train.reshape(X_train.shape[0], -1)
X_test_reshape = X_test.reshape(X_test.shape[0], -1)

In [12]:
rf_clf = RandomForestClassifier(n_estimators=100, min_samples_split=5, random_state=42)
rf_clf.fit(X_train_reshape, y_train)

RandomForestClassifier(min_samples_split=5, random_state=42)

In [13]:
y_pred = rf_clf.predict(X_test_reshape)

In [14]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión del árbol de decisión: {accuracy:.4f} ({accuracy*100:.2f}%)")

Precisión del árbol de decisión: 0.6408 (64.08%)


In [15]:
print(X_train.shape[1:])

(64, 64, 3)


## RED NEURONAL CONVENCIONAL

In [16]:
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape=X_train.shape[1:]))  
model.add(keras.layers.BatchNormalization())
# model.add(keras.layers.Dense(1500, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(500, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(1, activation='sigmoid', kernel_initializer='glorot_normal'))

/home/ciabd14/anaconda3/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
W0000 00:00:1776700213.699782   13689 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [17]:
model.compile(loss='binary_crossentropy', optimizer=keras.optimizers.Adam(learning_rate=0.01),metrics=['accuracy'])

In [18]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=1000, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/1000
633/633 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.5985 - loss: 0.6738 - val_accuracy: 0.6191 - val_loss: 0.6558
Epoch 2/1000
633/633 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.6404 - loss: 0.6282 - val_accuracy: 0.6636 - val_loss: 0.6226
Epoch 3/1000
633/633 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.6638 - loss: 0.6053 - val_accuracy: 0.6356 - val_loss: 0.6620
Epoch 4/1000
633/633 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.6810 - loss: 0.5911 - val_accuracy: 0.6560 - val_loss: 0.6198
Epoch 5/1000
633/633 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.6957 - loss: 0.5739 - val_accuracy: 0.6551 - val_loss: 0.6362
Epoch 6/1000
633/633 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.7086 - loss: 0.5591 - val_accuracy: 0.6449 - val_loss: 0.6385
Epoch 7/1000
633/633 ━━━━━━━━━━━━━━━━━━━━ 19s 29ms/step - accuracy: 0.7239 - loss: 0.5416 - val_accuracy: 0.6542 - val_loss: 0.6269
Epoch 8/1000
633/633 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.7382 -

In [19]:
model.evaluate(X_test, y_test)

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6504 - loss: 0.6279


[0.627901554107666, 0.6503999829292297]

## RED NEURONAL CONVOLUCIONAL

In [20]:
X = photos
y = labels

X = X/255.0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [21]:
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten

model = keras.models.Sequential()
model.add(Conv2D(32,(3,3), activation='relu', input_shape=X_train.shape[1:]))
model.add(MaxPool2D(2,2))
model.add(Conv2D(64,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(128,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Flatten())
model.add(keras.layers.Dense(128,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(28,activation='relu',kernel_initializer='he_normal'))
# model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(1,activation='sigmoid',kernel_initializer='glorot_normal'))

/home/ciabd14/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
model.compile(loss='crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=1000,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/1000
80/80 ━━━━━━━━━━━━━━━━━━━━ 10s 108ms/step - accuracy: 0.5675 - loss: 0.6799 - val_accuracy: 0.6107 - val_loss: 0.6562
Epoch 2/1000
80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 108ms/step - accuracy: 0.6525 - loss: 0.6279 - val_accuracy: 0.6676 - val_loss: 0.6092
Epoch 3/1000
80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 106ms/step - accuracy: 0.6811 - loss: 0.5959 - val_accuracy: 0.6991 - val_loss: 0.5805
Epoch 4/1000
80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 107ms/step - accuracy: 0.7066 - loss: 0.5674 - val_accuracy: 0.6956 - val_loss: 0.5712
Epoch 5/1000
80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 111ms/step - accuracy: 0.7258 - loss: 0.5478 - val_accuracy: 0.7160 - val_loss: 0.5514
Epoch 6/1000
80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 109ms/step - accuracy: 0.7445 - loss: 0.5230 - val_accuracy: 0.6880 - val_loss: 0.5863
Epoch 7/1000
80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 106ms/step - accuracy: 0.7486 - loss: 0.5124 - val_accuracy: 0.7622 - val_loss: 0.5067
Epoch 8/1000
80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 108ms/step - accuracy: 0.7618 - loss: 0.4985 -

In [23]:
model.evaluate(X_test,y_test)

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8136 - loss: 0.4089


[0.40885084867477417, 0.8136000037193298]

In [24]:
# model.save("model.keras")